# C8 · Interpolation: a rainfall map from 24 gauges

> **Time:** about 2 hours  
> **You need:** Parts A and B, and Lessons C1 to C7, especially functions (A5), UTM 45N (B4), plotting with `extent=` (C2), coarsening with `reproject` (C4) and `rasterize` and `zonal_stats` (C7).  
> **By the end you can:**
> 1. explain what interpolation is, and make **nearest-neighbour** (Voronoi) and **inverse distance weighting** (IDW) surfaces, with IDW written as your own function,
> 2. fit an **ordinary kriging** model with `pykrige`, and read its variogram and its map of uncertainty,
> 3. test every method honestly with **leave-one-out** validation and **RMSE**, never on the points used for fitting, and use elevation as a helper.

Rain gauges measure rainfall at a few points. Flood, landslide and water-supply studies need rainfall everywhere, in every cell of a grid. This lesson turns points into a surface in four ways, and then asks the most important question: *how good is each map?*

The 24 gauges in `practice_rain_gauges.csv` are **made-up practice data** (see `data/README.md`). Rainfall was set to rise with elevation, plus random noise. The values are not measurements, and the gauges are not real stations.

**Counting lines:** as in Part B, lines that only set up a plot (`fig, ax = plt.subplots()` and `plt.show()`) do not count toward the Q1/Q2/Q3 line counts.

**About the numbers:** kriging fits its variogram with a numerical search, which can end in a slightly different place with other library versions. Your kriging results may then differ from the text by a few mm, or a few hundred metres of range.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 32 of 39 · [Course home](../README.md)  
← Previous: [C7 · Vector and raster together: landslides, districts and slope](C07_Vector_and_raster_together.ipynb) · Next: [C9 · Spatial autocorrelation: are neighbours alike?](C09_Spatial_autocorrelation.ipynb) →

In [ ]:
# Optional: marks your answers as you go. Delete this cell to turn it off.
%run ../tools/tutor_on.py

## 1. Why interpolate: from 24 gauges to a grid

**Interpolation** estimates values at places where nothing was measured, from the places where something was. Here: annual rainfall in every cell of a grid, from 24 gauges.

Every method rests on one idea: places close together tend to be more alike than places far apart. Geographers call this the **first law of geography**.

Interpolation works with distances, so work in metres. Convert the gauges to UTM 45N, as in B4. In degrees, a step east is shorter on the ground than a step north (B1), and distances would be distorted.

The grid to fill is the DEM, reprojected to UTM 45N with **500 m** cells (C4). `Resampling.average` gives each big cell the mean of the small cells inside it, the right choice when cells grow much larger. A coarse grid keeps the lesson fast: about 12,000 cells instead of 3 million.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import rioxarray as rxr
from matplotlib.colors import ListedColormap
from rasterio.enums import Resampling

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)

rain = pd.read_csv(DATA / "practice_rain_gauges.csv")          # made-up practice data
gauges_utm = gpd.GeoDataFrame(rain, geometry=gpd.points_from_xy(rain["lon"], rain["lat"]),
                              crs="EPSG:4326").to_crs(32645)
gauges_utm["x_m"] = gauges_utm.geometry.x
gauges_utm["y_m"] = gauges_utm.geometry.y
print(gauges_utm[["gauge", "elevation_m", "annual_rain_mm", "x_m", "y_m"]].head(3).round(0))
print(gauges_utm["annual_rain_mm"].describe().round(1))

In [ ]:
dem = rxr.open_rasterio(DATA / "dem_kathmandu_melamchi.tif", masked=True).squeeze("band", drop=True)
dem_500 = dem.rio.reproject("EPSG:32645", resolution=500, resampling=Resampling.average)
valid = np.isfinite(dem_500.values)              # True where the grid has data
print(dem_500.shape, dem_500.rio.resolution(), valid.sum(), "cells with data")

left, bottom, right, top = dem_500.rio.bounds()
extent = [left, right, bottom, top]

land_cmap = ListedColormap(plt.colormaps["terrain"](np.linspace(0.25, 1, 256)))   # as in C2: "terrain" without its sea blues
fig, ax = plt.subplots(figsize=(9, 6))   # fig: the whole figure; ax: the map area inside it (A10)
im = ax.imshow(dem_500.values, extent=extent, cmap=land_cmap)
dots = ax.scatter(gauges_utm["x_m"], gauges_utm["y_m"], c=gauges_utm["annual_rain_mm"],
                  cmap="Blues", edgecolor="black", s=70)
fig.colorbar(im, ax=ax, label="Elevation (m)", shrink=0.7)
fig.colorbar(dots, ax=ax, label="Annual rainfall (mm, made-up)", shrink=0.7)
ax.set_title("24 practice rain gauges on the 500 m DEM")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.show()

The made-up gauges record between 931 and 2,005 mm a year, 1,404.3 mm on average. The 500 m grid has 102 rows and 120 columns; 11,990 of its 12,240 cells have data, and the rest are the empty slivers along the edges that you met in C4 and C7.

The darkest blue dots tend to sit on the brown, higher ground. Keep that in mind: step 7 uses it.

`land_cmap` is the colour map of C2, step 4: `"terrain"` without its sea blues, which would paint the deep valleys like lakes and clash with the blue dots.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA). The full licence notice is in `data/README.md`.*

### Practice

**Q1 · one line:** How many cells does `dem_500` hold in total? Use the attribute from A8 that counts all values.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
dem_500.size
```

12,240 cells: 102 rows × 120 columns. Only 24 of them contain a gauge, so interpolation has to estimate the other 99.8%.
</details>

**Q2 · two lines:** Which gauge is the wettest, and which the driest? Sort `gauges_utm` by `annual_rain_mm` and print the first and the last row, with the `gauge`, `elevation_m` and `annual_rain_mm` columns.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
by_rain = gauges_utm.sort_values("annual_rain_mm")[["gauge", "elevation_m", "annual_rain_mm"]]
print(by_rain.iloc[[0, -1]])
```

G21 is the driest (931 mm, at 914 m) and G06 the wettest (2,005 mm, at 2,289 m), which is also the highest gauge. `.iloc[[0, -1]]` takes the first and the last row in one go.
</details>

**Q3 · three lines:** How far is gauge G10 from its nearest neighbour? Take G10's point, measure the distance from every gauge to it with `.distance` (B8), leave out the zero (its distance to itself) and print the smallest, in km.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
g10 = gauges_utm.loc[gauges_utm["gauge"] == "G10", "geometry"].iloc[0]
dist_km = gauges_utm.distance(g10) / 1000
print(round(dist_km[dist_km > 0].min(), 1))
```

About 9.0 km. G10 lies in the north-east, with few gauges around it, so any surface will be less certain there. Distances in metres are correct here because `gauges_utm` is in UTM 45N.
</details>

## 2. Nearest neighbour: Voronoi polygons

The simplest method gives every place the value of its **nearest gauge**. The area that is closer to one gauge than to any other is that gauge's **Voronoi polygon**. In rainfall work they are also called **Thiessen polygons**, after the meteorologist who used them for exactly this.

- `gauges_utm.union_all()` joins the 24 points into one **MultiPoint** (B8).
- `shapely.voronoi_polygons(points, extend_to=box)` returns a collection of polygons that covers the box. `.geoms` lists them.
- The polygons come in no fixed order, so `gpd.sjoin` with `predicate="contains"` (B9) attaches each gauge's rainfall to the polygon that contains it.

Finally, `rasterize` (C7) burns the polygons into the 500 m grid, so that nearest neighbour becomes a raster like the other methods.

In [ ]:
import shapely
from shapely.geometry import box

study_area = box(left, bottom, right, top)             # the grid's rectangle (B2)
voronoi = shapely.voronoi_polygons(gauges_utm.union_all(), extend_to=study_area)
print(voronoi.geom_type, len(voronoi.geoms))

thiessen = gpd.GeoDataFrame(geometry=list(voronoi.geoms), crs=32645)
thiessen = gpd.clip(thiessen, study_area)              # cut to the rectangle (B7)
thiessen = gpd.sjoin(thiessen, gauges_utm[["gauge", "annual_rain_mm", "geometry"]], predicate="contains")
thiessen["area_km2"] = thiessen.area / 1000000
print(thiessen.sort_values("area_km2")[["gauge", "annual_rain_mm", "area_km2"]].iloc[[0, -1]].round(1))

In [ ]:
from rasterio import features

pairs = list(zip(thiessen.geometry, thiessen["annual_rain_mm"]))
nearest_mm = features.rasterize(pairs, out_shape=dem_500.shape, transform=dem_500.rio.transform(),
                                fill=np.nan, dtype="float32")
nearest_mm = np.where(valid, nearest_mm, np.nan)       # keep the DEM's footprint only (C5)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(nearest_mm, extent=extent, cmap="Blues", vmin=900, vmax=2050)
thiessen.boundary.plot(ax=ax, color="grey", linewidth=0.6)
ax.scatter(gauges_utm["x_m"], gauges_utm["y_m"], color="black", s=12)
ax.set_xlim(left, right)
ax.set_ylim(bottom, top)
fig.colorbar(im, ax=ax, label="Annual rainfall (mm, made-up)", shrink=0.8)
ax.set_title("Nearest neighbour: one value per Voronoi polygon")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.show()

The collection holds 24 polygons, one per gauge. After clipping, G05's polygon is the smallest (about 27.3 km²) and G10's the largest (about 384.6 km²): G10 alone decides the rainfall of a large part of the north-east.

The map is a patchwork. Rainfall jumps by hundreds of millimetres at each polygon edge, which real rainfall does not do. Still, Thiessen polygons are a classic way to turn gauge values into an average for a whole area, as Q2 shows.

`fill=np.nan` needs a float type, so the grid is `float32` (C7: the fill must fit the `dtype`).

### Practice

**Q1 · one line:** How many rows does `thiessen` have after the join? Print its length.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(len(thiessen))
```

24: every polygon contains exactly one gauge, so `sjoin` keeps all of them.
</details>

**Q2 · two lines:** A **Thiessen mean** weights each gauge by the area of its polygon. Work it out for the rectangle, and print it next to the plain mean of the 24 gauges.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
thiessen_mean_mm = (thiessen["annual_rain_mm"] * thiessen["area_km2"]).sum() / thiessen["area_km2"].sum()
print(round(thiessen_mean_mm, 1), round(gauges_utm["annual_rain_mm"].mean(), 1))
```

About 1,505.6 mm against a plain mean of 1,404.3 mm. Lonely gauges such as G10 (1,610 mm) cover large polygons and so count for more. Gauges packed close together share their area and count for less.
</details>

**Q3 · three lines:** How much of the rectangle gets more than 1,500 mm under nearest neighbour? Filter `thiessen`, add up the areas, and print the km² and the percentage of the whole rectangle.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
wet = thiessen[thiessen["annual_rain_mm"] > 1500]
wet_km2 = wet["area_km2"].sum()
print(round(wet_km2, 1), round(wet_km2 / thiessen["area_km2"].sum() * 100, 1))
```

About 1,715.1 km², or 56.0% of the rectangle. With polygons, "how much area" questions are sums of polygon areas, as in Part B.
</details>

## 3. Inverse distance weighting, written as a function

Nearest neighbour listens to one gauge only. **Inverse distance weighting** (IDW) listens to all of them, the near ones loudest.

Each gauge gets a **weight** w = 1 / dᵖ, where d is its distance from the place you estimate and p is the **power**. The estimate is the **weighted mean**: the sum of w × value, divided by the sum of w. Halve the distance, and with p = 2 the weight becomes four times larger.

At a gauge itself d = 0, and 1 / 0 has no answer. The function below returns that gauge's own value instead. So IDW **honours** the measurements: it is an **exact interpolator**.

You write IDW as a function (A5) for one place, with NumPy arrays (A8) holding all 24 gauges at once. A second function calls it for every cell of the grid (functions calling functions, A5).

**Predict:** what will `idw` return at the exact position of gauge G01, which recorded 1,574 mm?

In [ ]:
xs_m = gauges_utm["x_m"].to_numpy()
ys_m = gauges_utm["y_m"].to_numpy()
rain_mm = gauges_utm["annual_rain_mm"].to_numpy().astype(float)

def idw(x_m, y_m, xs_m, ys_m, values, power=2):
    """Inverse distance weighted estimate at one place (x_m, y_m)."""
    dist_m = np.sqrt((xs_m - x_m) ** 2 + (ys_m - y_m) ** 2)   # distance to every gauge
    if dist_m.min() == 0:                  # exactly on a gauge: return its value
        return values[dist_m.argmin()]
    weights = 1 / dist_m ** power
    return (weights * values).sum() / weights.sum()

# Kathmandu Durbar Square (approximate), converted to UTM 45N as in B4
durbar_utm = gpd.GeoSeries(gpd.points_from_xy([85.307], [27.704]), crs=4326).to_crs(32645)
durbar_x_m, durbar_y_m = durbar_utm.x.iloc[0], durbar_utm.y.iloc[0]

print(idw(xs_m[0], ys_m[0], xs_m, ys_m, rain_mm))               # on gauge G01
print(round(idw(durbar_x_m, durbar_y_m, xs_m, ys_m, rain_mm), 1))  # Durbar Square

In [ ]:
def idw_surface(xs_m, ys_m, values, power=2):
    """IDW estimate for every cell of dem_500 with data; NaN elsewhere."""
    grid_x = dem_500.x.values                # x of each column's centre
    grid_y = dem_500.y.values                # y of each row's centre, north to south
    surface = np.full(dem_500.shape, np.nan) # a grid of NaN to fill in
    for row in range(len(grid_y)):
        for col in range(len(grid_x)):
            if valid[row, col]:
                surface[row, col] = idw(grid_x[col], grid_y[row], xs_m, ys_m, values, power)
    return surface

idw2_mm = idw_surface(xs_m, ys_m, rain_mm)
print(round(np.nanmin(idw2_mm), 1), round(np.nanmax(idw2_mm), 1), round(np.nanmean(idw2_mm), 1))

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(idw2_mm, extent=extent, cmap="Blues", vmin=900, vmax=2050)
ax.scatter(gauges_utm["x_m"], gauges_utm["y_m"], color="black", s=12)
fig.colorbar(im, ax=ax, label="Annual rainfall (mm, made-up)", shrink=0.8)
ax.set_title("Inverse distance weighting, power 2")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.show()

On G01 the function returns exactly 1,574.0 mm, the gauge's own value. At Durbar Square, between several gauges, IDW estimates about 1,515.6 mm.

`np.full(shape, np.nan)` makes a grid of the given shape filled with `NaN`; the loops then fill the cells with data, row by row, as in A3. The surface runs from about 933.3 to 2,002.6 mm, always inside the range of the gauges: a weighted mean can never be larger than the largest value, or smaller than the smallest.

The map is smooth, but each gauge sits in a small ring of its own colour, a **bull's-eye**. That is the typical look of IDW.

### Practice

**Q1 · one line:** Estimate the rainfall at Durbar Square with `power=1` instead of 2.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
idw(durbar_x_m, durbar_y_m, xs_m, ys_m, rain_mm, power=1)
```

About 1,460.0 mm, closer to the mean of all gauges (1,404.3 mm). With a lower power, far gauges count for more.
</details>

**Q2 · two lines:** Read the IDW estimate at a gauge from the **grid**, not from the function: find the row and column of gauge G10 with `rasterio.transform.rowcol` (C1) and `dem_500.rio.transform()`, then print `idw2_mm` at that cell next to G10's measured value (index 9 in the arrays).

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
row, col = rasterio.transform.rowcol(dem_500.rio.transform(), xs_m[9], ys_m[9])
print(round(idw2_mm[row, col], 1), rain_mm[9])
```

1,610.0 mm in the grid and 1,610.0 mm measured. The cell's centre is not exactly on the gauge, but so close to it that G10's weight swamps all the others. Cells farther from G10 blend in more of the other gauges.
</details>

**Q3 · three lines:** Where do IDW and nearest neighbour disagree most? Make the absolute difference of `idw2_mm` and `nearest_mm`, find its largest value with `np.nanmax`, and its row and column with `np.nanargmax` and `np.unravel_index` (A8).

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
diff_mm = np.abs(idw2_mm - nearest_mm)
row, col = np.unravel_index(np.nanargmax(diff_mm), diff_mm.shape)
print(round(np.nanmax(diff_mm), 1), row, col)
```

About 658.2 mm, at row 44, column 76. That cell lies in the Voronoi polygon of G06, the wettest gauge, near its edge with drier gauges. Nearest neighbour copies G06's 2,005 mm there; IDW blends it with its drier neighbours.
</details>

## 4. Choosing the power

The power p decides how fast a gauge's influence fades with distance.

- **p = 1**: far gauges still count a lot. The surface is smooth and pulled towards the mean of all gauges.
- **p = 2**: the most common choice, and the default of many GIS programs.
- **Large p**: the nearest gauge takes almost all the weight, and the map turns into the nearest-neighbour patchwork.

All three are "correct" IDW maps. Looking at them cannot tell you which is best; step 5 measures it.

**Predict:** of the three maps below, which one will look most like the Voronoi map of step 2?

In [ ]:
idw1_mm = idw_surface(xs_m, ys_m, rain_mm, power=1)
idw4_mm = idw_surface(xs_m, ys_m, rain_mm, power=4)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, surface, power in zip(axes, [idw1_mm, idw2_mm, idw4_mm], [1, 2, 4]):
    im = ax.imshow(surface, extent=extent, cmap="Blues", vmin=900, vmax=2050)
    ax.scatter(gauges_utm["x_m"], gauges_utm["y_m"], color="black", s=8)
    ax.set_title(f"IDW, power {power}")
    ax.set_xticks([])
    ax.set_yticks([])
    print(power, "standard deviation of the surface:", round(np.nanstd(surface), 1), "mm")
fig.colorbar(im, ax=axes, label="Annual rainfall (mm, made-up)", shrink=0.8)
plt.show()

With power 1 the map is soft and mostly middle blue: the **standard deviation** of its cells (how much they spread around their mean, A9) is only about 82.1 mm. With power 2 it is about 171.4 mm, and with power 4 about 246.1 mm, with sharp patches around each gauge: the most Voronoi-like of the three.

The same colour limits (`vmin`, `vmax`) on all three maps make them comparable, so one colour bar serves all. `ax.set_xticks([])` hides the axis numbers to save space; the grid is the same 500 m UTM grid as before.

### Practice

**Q1 · one line:** With power 2, how much more weight does a gauge 2 km away get than a gauge 4 km away? Work it out with the formula w = 1 / dᵖ.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
(1 / 2 ** 2) / (1 / 4 ** 2)
```

4.0: half the distance, four times the weight. Units cancel in the ratio, so km work as well as metres here.
</details>

**Q2 · two lines:** Work out the same ratio, 2 km against 4 km, for power 1 and for power 4.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print((1 / 2 ** 1) / (1 / 4 ** 1))
print((1 / 2 ** 4) / (1 / 4 ** 4))
```

2.0 for power 1 and 16.0 for power 4. Each step up in power doubles the advantage of the nearer gauge again, which is why large powers end in patches.
</details>

**Q3 · three lines:** Make an IDW surface with power 8 and count the cells with data where it lies within 50 mm of `nearest_mm`. Print the count and the percentage of the 11,990 cells with data.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
idw8_mm = idw_surface(xs_m, ys_m, rain_mm, power=8)
close = np.abs(idw8_mm - nearest_mm)[valid] <= 50
print(close.sum(), round(close.mean() * 100, 1))
```

8,413 cells, about 70.2%, are within 50 mm. With power 8, IDW is close to nearest neighbour almost everywhere except near the polygon edges, where it still blends.
</details>

## 5. Testing honestly: leave-one-out and RMSE

An exact interpolator returns each gauge's own value, so checking it **at** the gauges proves nothing. An honest test predicts gauges the method has **not** seen.

**Leave-one-out cross-validation** does this with every gauge in turn: hide gauge 1, estimate it from the other 23, note the **error** (estimate − measured value); then hide gauge 2, and so on, 24 times.

The **RMSE** (root mean square error) sums up the 24 errors: square them, take the mean, then the square root. It is a typical size of error, in mm, with big misses counting extra. The **mean error** shows **bias**: whether a method guesses too high or too low on average.

Always compare with a **baseline**, a method that ignores location: here, the mean of the other 23 gauges. A map is only worth drawing if it beats the baseline. `keep = np.arange(24) != i` is a mask (A8) that is `True` for every gauge except number `i`.

In [ ]:
def rmse(errors):
    """Root mean square error of an array of errors."""
    return np.sqrt(np.mean(errors ** 2))

def loo_errors(power):
    """Leave-one-out errors (mm) of IDW with the given power."""
    errors_mm = []
    for i in range(len(rain_mm)):
        keep = np.arange(len(rain_mm)) != i          # every gauge except number i
        estimate = idw(xs_m[i], ys_m[i], xs_m[keep], ys_m[keep], rain_mm[keep], power)
        errors_mm.append(estimate - rain_mm[i])
    return np.array(errors_mm)

baseline_mm = np.array([rain_mm[np.arange(24) != i].mean() - rain_mm[i] for i in range(24)])
print(f"baseline (mean of the others): RMSE {rmse(baseline_mm):.1f} mm")
for power in [1, 2, 3, 4, 8]:
    errors_mm = loo_errors(power)
    print(f"IDW power {power}: RMSE {rmse(errors_mm):.1f} mm, mean error {errors_mm.mean():.1f} mm")

The baseline misses by about 361.3 mm. IDW does best with power 2 (about 311.0 mm), then power 1 (322.5 mm) and power 3 (328.3 mm). Power 4 (342.4 mm) and power 8 (359.4 mm) are worse, and power 8 is hardly better than the baseline.

Every mean error is negative: on average IDW guesses too low, by about 69.8 mm with power 2. So the best IDW map beats "use the average everywhere" by only about 14%. Distance alone explains these rainfall values poorly. Step 7 finds out why.

Be modest with such numbers: 24 gauges are few, and moving one gauge could reorder powers whose RMSEs are only 10 mm apart.

### Practice

**Q1 · one line:** Which gauge does IDW with power 2 miss by the most? Use `np.abs(...).argmax()` on `loo_errors(2)` to pick the row of `gauges_utm["gauge"]`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
gauges_utm["gauge"].iloc[np.abs(loo_errors(2)).argmax()]
```

G17, the northernmost gauge and one of the wettest. Hidden, it can only be estimated from gauges to its south, most of them drier, so any average of them falls short.
</details>

**Q2 · two lines:** Store the power-2 leave-one-out errors in a new column `loo_err_mm` of `gauges_utm`, then print the mean error of the gauges above 1,700 m.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
gauges_utm["loo_err_mm"] = loo_errors(2)
print(round(gauges_utm.loc[gauges_utm["elevation_m"] > 1700, "loo_err_mm"].mean(), 1))
```

About −336.4 mm: IDW underestimates the high gauges badly. Their neighbours are mostly lower and drier, and IDW cannot know that height matters.
</details>

**Q3 · three lines:** Draw a scatter plot (A10) of `elevation_m` (x) against `loo_err_mm` (y), with a horizontal line at 0 (`ax.axhline(0)`) and labelled axes.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(gauges_utm["elevation_m"], gauges_utm["loo_err_mm"])
ax.axhline(0, color="grey")
ax.set_xlabel("Gauge elevation (m)")
ax.set_ylabel("Leave-one-out error (mm)")
plt.show()
```

The points fall from upper left to lower right: low gauges are overestimated, high gauges underestimated. Errors with a clear pattern are a hint that a helper variable, here elevation, is missing.
</details>

## 6. Kriging, tested on gauges it has not seen

**Kriging** is also a weighted mean, but it learns its weights from the data. First it measures how different two gauges are, on average, as the distance between them grows. Half the mean squared difference is the **semivariance**; plotted against distance it is the **variogram**. A curve fitted to it has three numbers:

- **nugget**: the difference left even between gauges very close together (noise, very local effects), in mm²,
- **range**: the distance beyond which gauges are no more alike than any two random gauges,
- **partial sill**: how far the curve rises from the nugget to its plateau, in mm².

`OrdinaryKriging(xs, ys, values, variogram_model="spherical")` from `pykrige` fits the curve. `.execute("grid", grid_x, grid_y)` fills a grid; `.execute("points", xs, ys)` estimates at points. Both also return the **kriging variance**: an uncertainty for every estimate.

The honest rule from step 5 still holds: test only on gauges the model has **not** seen. A common mistake is to fit kriging on all points and then test it on some of those same points. The third cell makes that mistake on purpose.

In [ ]:
from pykrige.ok import OrdinaryKriging

ok = OrdinaryKriging(xs_m, ys_m, rain_mm, variogram_model="spherical")
psill, range_m, nugget = ok.variogram_model_parameters
print(f"partial sill {psill:.0f} mm², range {range_m / 1000:.1f} km, nugget {nugget:.0f} mm²")

dist_m = np.linspace(0, ok.lags.max(), 100)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(ok.lags / 1000, ok.semivariance, "o", label="From the gauges")
ax.plot(dist_m / 1000, ok.variogram_function(ok.variogram_model_parameters, dist_m), label="Fitted spherical curve")
ax.set_xlabel("Distance between gauges (km)")
ax.set_ylabel("Semivariance (mm²)")
ax.legend()
plt.show()

In [ ]:
krig_mm, krig_var = ok.execute("grid", dem_500.x.values, dem_500.y.values)
krig_mm = np.where(valid, krig_mm, np.nan)
krig_sd_mm = np.where(valid, np.sqrt(krig_var), np.nan)     # standard deviation, in mm
print(krig_mm.shape, round(np.nanmin(krig_mm), 1), round(np.nanmax(krig_mm), 1))
print(round(np.nanmin(krig_sd_mm), 1), round(np.nanmax(krig_sd_mm), 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im0 = axes[0].imshow(krig_mm, extent=extent, cmap="Blues", vmin=900, vmax=2050)
im1 = axes[1].imshow(krig_sd_mm, extent=extent, cmap="Oranges")
for ax in axes:
    ax.scatter(gauges_utm["x_m"], gauges_utm["y_m"], color="black", s=10)
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
fig.colorbar(im0, ax=axes[0], label="Annual rainfall (mm, made-up)", shrink=0.8)
fig.colorbar(im1, ax=axes[1], label="Kriging standard deviation (mm)", shrink=0.8)
axes[0].set_title("Ordinary kriging estimate")
axes[1].set_title("Its uncertainty")
plt.show()

In [ ]:
krig_errors_mm = []
for i in range(len(rain_mm)):
    keep = np.arange(len(rain_mm)) != i                 # train on the other 23 only
    ok_23 = OrdinaryKriging(xs_m[keep], ys_m[keep], rain_mm[keep], variogram_model="spherical")
    estimate, variance = ok_23.execute("points", xs_m[i:i + 1], ys_m[i:i + 1])
    krig_errors_mm.append(estimate[0] - rain_mm[i])
krig_errors_mm = np.array(krig_errors_mm)
print("kriging, leave-one-out RMSE:", round(rmse(krig_errors_mm), 1), "mm")

# THE MISTAKE, ON PURPOSE: "testing" on the gauges the model was fitted on
same_mm, same_var = ok.execute("points", xs_m, ys_m)
print("kriging, tested on its own gauges:", round(rmse(same_mm - rain_mm), 1), "mm")

The fitted curve has a nugget of about 15,168 mm², a partial sill of about 116,041 mm² and a range of about 17.7 km. Gauges closer than about 18 km are more alike than gauges farther apart. The dots of the variogram scatter a lot: with 24 gauges there are few pairs at each distance, so the fitted curve is a rough guess.

The grid comes back with one row per y value, in the order you gave them. `dem_500.y` runs north to south, so the result is north-up (C1). The estimate ranges from about 967.1 to 1,898.3 mm, a narrower range than IDW's: with a nugget, kriging does not force a sharp bull's-eye at each gauge. The standard deviation is about 165.0 mm next to the gauges and up to about 377.8 mm far from them, along the northern edge. **That second map is kriging's real gift: it shows where the first map can be trusted.**

Refitted 24 times on 23 gauges, kriging scores an RMSE of about 307.0 mm: a little better than IDW (311.0 mm), and still far from perfect. "Tested" on its own gauges it scores 0.0 mm, a perfect and meaningless score, because kriging is an exact interpolator too. Any error measured on the fitting points says nothing about the map.

### Practice

**Q1 · one line:** The nugget is in mm². Print its square root, in mm, and the range in km, both rounded to 1 decimal.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round(np.sqrt(nugget), 1), round(range_m / 1000, 1))
```

About 123.2 mm and 17.7 km. The square root of the nugget is the size of the purely local variation in this model: noise that no interpolation can remove.
</details>

**Q2 · two lines:** Estimate the rainfall at Durbar Square with `ok.execute("points", ...)`, using `[durbar_x_m]` and `[durbar_y_m]` (lists of one value). Print the estimate and its standard deviation.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
durbar_mm, durbar_var = ok.execute("points", [durbar_x_m], [durbar_y_m])
print(round(float(durbar_mm[0]), 1), round(float(np.sqrt(durbar_var[0])), 1))
```

About 1,525.1 mm, give or take about 332.6 mm (one standard deviation). IDW said about 1,515.6 mm. The standard deviation is close to the largest on the whole map (377.8 mm): no gauge stands near Durbar Square, and kriging says so.
</details>

**Q3 · three lines:** Fit the same leave-one-out test with `variogram_model="linear"`, a straight-line variogram, and print its RMSE. Loop as in the third example cell.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
lin_errors = []
for i in range(len(rain_mm)):
    keep = np.arange(len(rain_mm)) != i
    ok_lin = OrdinaryKriging(xs_m[keep], ys_m[keep], rain_mm[keep], variogram_model="linear")
    lin_errors.append(ok_lin.execute("points", xs_m[i:i + 1], ys_m[i:i + 1])[0][0] - rain_mm[i])
print(round(rmse(np.array(lin_errors)), 1))
```

About 357.9 mm, hardly better than the baseline (361.3 mm). The choice of variogram model matters, and leave-one-out tells you which one to trust. `.execute(...)[0][0]` takes the estimates, then the first one.
</details>

## 7. Elevation as a helper

Distance-only methods beat the baseline by about 15% at most. Step 5 showed why: the errors depend on **elevation**. In the hills of Nepal, rainfall depends strongly on height and on which slopes face the monsoon winds. These made-up data were built from elevation alone, plus random noise.

The gauges carry their elevation, and the DEM knows the elevation of every cell. `np.polyfit(x, y, 1)` fits the best straight line, y = a × x + b, and returns the slope `a` and the intercept `b`. Applied to every cell of `dem_500`, the line gives a **trend surface**.

The test is the same as before: leave one gauge out, fit the line on the other 23, and predict the hidden one.

Be honest about two things. The made-up data were generated from elevation, so this test is set up to succeed; real gauges would need checking. And the line knows only the elevations of the gauges: above the highest one it is **extrapolating**, guessing beyond the data.

In [ ]:
elev_m = gauges_utm["elevation_m"].to_numpy().astype(float)
a_mm_per_m, b_mm = np.polyfit(elev_m, rain_mm, 1)
print(f"rainfall = {a_mm_per_m:.3f} x elevation + {b_mm:.1f}")

line_m = np.array([elev_m.min(), elev_m.max()])
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(elev_m, rain_mm)
ax.plot(line_m, a_mm_per_m * line_m + b_mm, color="crimson", label="Fitted line")
ax.set_xlabel("Gauge elevation (m)")
ax.set_ylabel("Annual rainfall (mm, made-up)")
ax.legend()
plt.show()

In [ ]:
trend_errors_mm = []
for i in range(len(rain_mm)):
    keep = np.arange(len(rain_mm)) != i
    a, b = np.polyfit(elev_m[keep], rain_mm[keep], 1)      # fitted without gauge i
    trend_errors_mm.append(a * elev_m[i] + b - rain_mm[i])
print("elevation trend, leave-one-out RMSE:", round(rmse(np.array(trend_errors_mm)), 1), "mm")

In [ ]:
trend_mm = a_mm_per_m * dem_500.values + b_mm          # NaN cells stay NaN
print(round(np.nanmin(trend_mm), 1), round(np.nanmax(trend_mm), 1))
print(round(float(np.nanmax(dem_500.values)), 1), "m: the highest 500 m cell")

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(trend_mm, extent=extent, cmap="Blues", vmin=900, vmax=2050)
ax.scatter(gauges_utm["x_m"], gauges_utm["y_m"], color="black", s=12)
fig.colorbar(im, ax=ax, label="Annual rainfall (mm, made-up)", shrink=0.8, extend="max")
ax.set_title("Elevation trend surface")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.show()

The fitted line adds about 0.719 mm of rain a year per metre of height: about 72 mm per 100 m. Its leave-one-out RMSE is about 126.4 mm, against 307.0 mm for kriging and 361.3 mm for the baseline. Here a helper variable does far more than any choice of distance method.

The trend map follows the terrain: dry valleys, wet ridges, and detail everywhere, even where no gauge stands. It runs from about 766.8 to 3,160.3 mm. The top of that range is an extrapolation: the highest gauge is at 2,289 m, but the 500 m grid reaches about 3,863.7 m. `extend="max"` adds a pointed end to the colour bar to show that some cells are wetter than its top colour.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA). The full licence notice is in `data/README.md`.*

### Practice

**Q1 · one line:** What does the fitted line give for 1,400 m, the approximate elevation of Kathmandu on the facts sheet?

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
a_mm_per_m * 1400 + b_mm
```

About 1,387.7 mm (made-up model). One line of arithmetic, because the line is a formula.
</details>

**Q2 · two lines:** How much of the grid lies above the highest gauge, so that the trend surface extrapolates there? Compare `dem_500.values` with `elev_m.max()`, keep the cells with data, and print the percentage.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
above = dem_500.values[valid] > elev_m.max()
print(round(above.mean() * 100, 1))
```

About 8.5% of the cells lie above 2,289 m. There the map rests on the assumption that the line keeps rising, which no gauge confirms.
</details>

**Q3 · three lines:** The **residual** of a gauge is its measured rainfall minus the trend at its elevation. Compute the 24 residuals with the line fitted to all gauges, and print the largest one in absolute value and the gauge it belongs to.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
residual_mm = rain_mm - (a_mm_per_m * elev_m + b_mm)
biggest = np.abs(residual_mm).argmax()
print(gauges_utm["gauge"].iloc[biggest], round(residual_mm[biggest], 1))
```

G17 has the largest residual, about 250.0 mm: it is much wetter than its elevation suggests. It is also the gauge that IDW missed most in step 5. Residuals are what elevation does not explain; Review 2 tries to interpolate them.
</details>

## 8. When it breaks: `math` functions on arrays

In A5 you used `math.sqrt` for single numbers. Here it is used for the distances from one place to all 24 gauges at once. Run the cell.

In [ ]:
import math

dist_m = math.sqrt((xs_m - durbar_x_m) ** 2 + (ys_m - durbar_y_m) ** 2)

Read the last line: `TypeError: only 0-dimensional arrays can be converted to Python scalars`.

- A **scalar** is a single number. A **0-dimensional array** is NumPy's name for an array that holds only one number.
- `math.sqrt` takes one number only. It tried to turn the array of 24 squared distances into one number, and could not.

The `math` module from A5 works on single numbers; NumPy functions work on whole arrays at once (A8). The fix is `np.sqrt`, exactly as in the `idw` function. The same goes for `math.exp`, `math.log` and the others: use `np.exp`, `np.log` with arrays.

**Try it:** change `math.sqrt` to `np.sqrt` and run the cell again.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · A hold-out test

Leave-one-out is one honest test; a **hold-out** test is another. Put every fourth gauge aside as test gauges with `gauges_utm.iloc[::4]` and use the rest, `gauges_utm.drop(test.index)`, for training. Fit on the training gauges only, then print the RMSE on the test gauges of three methods: the training mean, IDW with power 2, and the elevation line.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
test = gauges_utm.iloc[::4]
train = gauges_utm.drop(test.index)
print(test["gauge"].tolist())
tx, ty, tv = train["x_m"].to_numpy(), train["y_m"].to_numpy(), train["annual_rain_mm"].to_numpy().astype(float)
obs_mm = test["annual_rain_mm"].to_numpy()

print("mean:", round(rmse(tv.mean() - obs_mm), 1))
idw_test = np.array([idw(x, y, tx, ty, tv) for x, y in zip(test["x_m"], test["y_m"])])
print("IDW: ", round(rmse(idw_test - obs_mm), 1))
a, b = np.polyfit(train["elevation_m"], tv, 1)
print("line:", round(rmse(a * test["elevation_m"].to_numpy() + b - obs_mm), 1))
```

The test gauges are G01, G05, G09, G13, G17 and G21. The training mean misses them by about 431.3 mm, IDW by about 372.8 mm and the elevation line by about 160.0 mm. The ranking matches leave-one-out, but every number is different: with only 6 test gauges, a hold-out score depends a lot on which gauges were set aside. Leave-one-out uses every gauge once, which is why it suits small data sets.
</details>

### Review 2 · Trend plus interpolated residuals

Combine steps 3 and 7: for each gauge left out, fit the line on the other 23, interpolate the 23 **residuals** with IDW (power 2) to the hidden gauge, and add the interpolated residual to the line's value. Print the leave-one-out RMSE and compare it with the line alone (126.4 mm).

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
combo_errors_mm = []
for i in range(len(rain_mm)):
    keep = np.arange(len(rain_mm)) != i
    a, b = np.polyfit(elev_m[keep], rain_mm[keep], 1)
    residuals = rain_mm[keep] - (a * elev_m[keep] + b)
    residual_here = idw(xs_m[i], ys_m[i], xs_m[keep], ys_m[keep], residuals)
    combo_errors_mm.append(a * elev_m[i] + b + residual_here - rain_mm[i])
print(round(rmse(np.array(combo_errors_mm)), 1))
```

About 153.3 mm: **worse** than the line alone. The made-up residuals are pure random noise, with no pattern in space, so interpolating them only adds noise. With real gauges the residuals often do have a pattern, for example wetter windward slopes, and then this combination can help. When kriging interpolates the residuals, the method is called **regression kriging**. The lesson: test every extra step; do not assume it helps.
</details>

### Review 3 · Rainfall per district

Use `zonal_stats` (C7) to find the mean of `trend_mm` in each district that touches the grid. Read the districts, select them with `.cx` on the DEM's box in degrees (85.2 to 85.8° E, 27.55 to 28.0° N), convert them to UTM 45N, and print a table of district and mean rainfall, wettest first. Remember the values are made-up.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
from rasterstats import zonal_stats

districts = gpd.read_file(DATA / "nepal_districts.geojson")
near = districts.cx[85.2:85.8, 27.55:28.0].to_crs(32645).reset_index(drop=True)
stats = zonal_stats(near, trend_mm, affine=dem_500.rio.transform(), nodata=np.nan, stats=["mean", "count"])
table = pd.DataFrame(stats)
table["district"] = near["district"]
print(table.sort_values("mean", ascending=False).round(1))
```

Makawanpur comes out wettest (about 1,676.2 mm), then Sindhupalchok (about 1,597.3 mm); Kabhrepalanchok is the driest (about 1,337.9 mm). But look at the `count` column: Makawanpur's mean rests on only 192 cells of 500 m, Rasuwa's on 141 and Dhading's on 117, small slivers at the edge of the grid. Report the count with every zonal mean.
</details>

### Review 4 · Save the trend map and read it back

Write `trend_mm` to `../outputs/c08_rain_trend.tif` as in C1: one band, `float32`, the CRS and transform of `dem_500`, and `nodata=np.nan`. Read it back and sample it at Durbar Square with `src.sample`. Use the UTM coordinates `durbar_x_m`, `durbar_y_m`, because the file is in UTM 45N (C7).

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
profile = {"driver": "GTiff", "height": trend_mm.shape[0], "width": trend_mm.shape[1], "count": 1,
           "dtype": "float32", "crs": dem_500.rio.crs, "transform": dem_500.rio.transform(), "nodata": np.nan}
with rasterio.open(OUT / "c08_rain_trend.tif", "w", **profile) as dst:
    dst.write(trend_mm.astype("float32"), 1)

with rasterio.open(OUT / "c08_rain_trend.tif") as src:
    print(src.crs, src.res, src.transform.e < 0)
    print(round(float(list(src.sample([(durbar_x_m, durbar_y_m)]))[0][0]), 1), "mm at Durbar Square")
```

The file is in EPSG:32645 with 500 m cells, and `transform.e < 0` is `True`: north is up (C1). Durbar Square's 500 m cell reads about 1,318.0 mm. That is lower than IDW's 1,515.6 mm: the line follows the low valley floor (C1 read 1,300 m at Durbar Square), while IDW only averages the gauges around it, wherever they stand.
</details>

### Review 5 · Where would a 25th gauge help most?

A new gauge helps most where the kriging map is least certain. Find the cell with the largest `krig_sd_mm` (`np.nanargmax` and `np.unravel_index`, A8), take its x and y from `dem_500.x.values` and `dem_500.y.values`, convert that point to longitude and latitude (B4), and print them with the standard deviation.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
row, col = np.unravel_index(np.nanargmax(krig_sd_mm), krig_sd_mm.shape)
x_m, y_m = dem_500.x.values[col], dem_500.y.values[row]
spot = gpd.GeoSeries(gpd.points_from_xy([x_m], [y_m]), crs=32645).to_crs(4326)
print(row, col, round(krig_sd_mm[row, col], 1), "mm")
print(round(spot.x.iloc[0], 3), round(spot.y.iloc[0], 3))
```

The most uncertain cell is at row 0, column 58, on the northern edge of the grid, near 85.49° E, 28.0° N, with a standard deviation of about 377.8 mm. The edges, especially the north, are far from every gauge. In practice you would also weigh access, safety and what the study needs, but the kriging variance is a sound place to start.
</details>

## Recap

- **Interpolation** estimates values between measured points. Work in a projected CRS (UTM 45N), because every method uses distances.
- **Nearest neighbour** gives Voronoi (Thiessen) patches; **IDW** is a weighted mean with weights 1 / dᵖ, smoother with a small power and patchier with a large one. Both are exact at the gauges.
- **Kriging** learns its weights from a **variogram** (nugget, range, partial sill) and also gives a **variance**: a map of where its estimate can be trusted.
- Test honestly: **leave-one-out** or **hold-out**, with **RMSE** and a **baseline**. Never test on the points used for fitting; that score is meaningless.
- A helper such as elevation can beat every distance method, but extrapolating beyond the data is a guess. With 24 made-up gauges, all results are rough.

### Words you learned

| Word | Meaning |
| --- | --- |
| interpolation | Estimating values at unmeasured places from measured ones |
| first law of geography | Near things are more alike than distant things |
| Voronoi / Thiessen polygon | The area closer to one point than to any other point |
| nearest neighbour | Interpolation that copies the value of the closest point |
| inverse distance weighting (IDW) | A weighted mean with weights 1 / dᵖ: nearer points count more |
| power (IDW) | The exponent p that sets how fast a point's weight falls with distance |
| weighted mean | A mean in which some values count more than others |
| exact interpolator | A method that returns the measured value at each measured point |
| bull's-eye | A ring of colour around a single point on an interpolated map |
| leave-one-out cross-validation | Hiding each point in turn and predicting it from the others |
| hold-out test | Setting some points aside for testing and fitting on the rest |
| RMSE | Root mean square error: the square root of the mean squared error |
| bias | A tendency to estimate too high or too low on average |
| baseline | A simple method that a better method must beat |
| kriging | Interpolation whose weights come from a fitted variogram |
| semivariance / variogram | Half the mean squared difference between points / its curve against distance |
| nugget, range, partial sill | The variogram's value near zero distance, the distance where it levels off, and its rise |
| kriging variance | Kriging's estimate of its own uncertainty in each cell |
| trend surface | A surface computed from a helper variable, here a line fitted to elevation |
| residual | A measured value minus the value a model predicts |
| extrapolation | Estimating beyond the range of the data |
| scalar | A single number |

## Next

The last lesson of Part C is **C9 · Spatial autocorrelation**: how to measure that nearby places are alike, the idea behind every method in this lesson, and why it matters for testing models on maps.

<!-- pygile-nav -->
**PyGILE Course** · lesson 32 of 39 · [Course home](../README.md)  
← Previous: [C7 · Vector and raster together: landslides, districts and slope](C07_Vector_and_raster_together.ipynb) · Next: [C9 · Spatial autocorrelation: are neighbours alike?](C09_Spatial_autocorrelation.ipynb) →